# q7: new vs returning visitors within each channel
checks if returning visitors buy more in every channel, not just overall (sql/10_channel_by_visitor.sql).
clean days only (nov 12 – jan 25, utc). rows with fewer than 30 purchases are flagged as small samples.


In [1]:
import warnings
warnings.filterwarnings("ignore", message="BigQuery Storage module not found")

from pathlib import Path
from google.cloud import bigquery

client = bigquery.Client(project="ga4-funnel-analysis-511719")
output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)


In [2]:
sql = """
select
    case
        when medium is null then 'unknown'
        when source in ('shop.googlemerchandisestore.com', 'googlemerchandisestore.com') then 'self-referral'
        when source = '(direct)' then 'direct'
        when medium = 'organic' then 'organic search'
        when medium = 'cpc' then 'paid search'
        when medium = 'referral' then 'referral'
        when medium = 'email' then 'email'
        when medium = 'affiliate' then 'affiliate'
        when medium in ('<Other>', '(data deleted)') then 'hidden'
        else 'other'
    end as channel,
    case
        when is_new_visitor then 'new'
        when not is_new_visitor then 'returning'
        else 'unknown'
    end as visitor_type,
    count(*) as sessions,
    countif(began_checkout or purchased) as reached_checkout,
    countif(purchased) as purchases,
    sum(revenue) as revenue
from `ga4-funnel-analysis-511719.ga4_analysis.session_base`
where date(timestamp_micros(session_start_time)) between date '2020-11-12' and date '2021-01-25'
group by channel, visitor_type
order by channel, visitor_type
"""

cvisitors = client.query(sql).to_dataframe()
cvisitors["conversion_rate"] = (cvisitors["purchases"] / cvisitors["sessions"]).round(4)
cvisitors["purchase_rate"] = (cvisitors["purchases"] / cvisitors["reached_checkout"]).round(4)
cvisitors["revenue_per_session"] = (cvisitors["revenue"] / cvisitors["sessions"]).round(2)
cvisitors["small_sample"] = cvisitors["purchases"] < 30
print(len(cvisitors), cvisitors["sessions"].sum(), cvisitors["purchases"].sum(), cvisitors["revenue"].sum())
cvisitors

18 299173 3968 304028.0


,channel,visitor_type,sessions,reached_checkout,purchases,revenue,conversion_rate,purchase_rate,revenue_per_session,small_sample
0,affiliate,new,426,10,1,23.0,0.0023,0.1,0.05,True
1,affiliate,returning,424,9,1,144.0,0.0024,0.1111,0.34,True
2,direct,new,27767,668,113,7380.0,0.0041,0.1692,0.27,False
3,direct,returning,5296,254,139,13602.0,0.0262,0.5472,2.57,False
4,email,new,15,9,6,630.0,0.4,0.6667,42.0,True
5,email,returning,72,19,13,951.0,0.1806,0.6842,13.21,True
6,hidden,new,20873,494,70,5138.0,0.0034,0.1417,0.25,False
7,hidden,returning,6669,366,194,13463.0,0.0291,0.5301,2.02,False
8,organic search,new,66048,2379,732,48856.0,0.0111,0.3077,0.74,False
9,organic search,returning,16982,1128,605,46115.0,0.0356,0.5363,2.72,False


In [3]:
cvisitors.to_csv(output_dir / "q7_channel_by_visitor.csv", index=False)